# 11 (Advanced) Sequence Forecasting: Predicting the Epidemic Curve with LSTM / CNN

In the first half of Ch11 we trained a classification model with PyTorch, and it tied with sklearn on 280 rows of data.
Here we switch to a task better suited to deep learning: **sequence forecasting**.

Workflow: **synthesize a dengue × temperature series → windowing → time-based train/val/test split → LSTM → 1D-CNN → compare against a naive baseline**

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Why is sequence forecasting different?

The earlier chapters (Ch03–Ch10) were mostly "cross-sectional" problems: each row of data is independent, and we use today's age, comorbidities, and exposure history to predict today's outcome.

**Sequence forecasting is different**: the data has a time order, and today's case count is correlated with yesterday's, the day before that, and so on (autocorrelation). Epidemiology also often has "leading indicators" — a variable that changes before the case count does, which amounts to an early warning.

### This section's scenario (synthetic teaching data)

Here we **do not use** the Legionella data from Pine and Cypress Nursing Home (that dataset doesn't have a long enough daily series). Instead we use a set of **synthetic "dengue × temperature" daily series**:

- **Temperature**: has seasonal fluctuation plus daily random noise, and is a **known, ahead-of-time-available** leading indicator
- **Case count**: driven jointly by "temperature 7 days ago" (the hotter it is, the more active the mosquito vector) and "the previous day's case count" (continuity of transmission), plus a weekly reporting rhythm and random noise

> ⚠️ This is **synthetic teaching data**, not real dengue surveillance data — the point is to learn the method of "how to use a sequence model to capture a leading indicator," not that these particular parameters carry epidemiological meaning.

**Task**: use the past 21 days of (case count, temperature) to predict the case count **7 days later**.

In [ ]:
# --- Step 1: Generate a synthetic "dengue × temperature" series ---
import pathlib

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (prevents Chinese labels from rendering as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

# -- Fix the random seed so results are reproducible on every run (a basic requirement of reproducible research, see Ch13) --
torch.manual_seed(1)
np.random.seed(1)

# -- Data generating process (DGP) --
n = 360                       # a bit over a year of daily data
t = np.arange(n)

# Temperature: seasonal sine wave + daily random noise (this is the "known, ahead-of-time-available" leading indicator)
temp = 24 + 7 * np.sin(2 * np.pi * (t - 30) / 365) + np.random.normal(0, 1.0, n)

# drive: the part of temperature above 24 degrees, representing "the hotter it is, the more active the mosquito vector"
drive = np.clip(temp - 24, 0, None)

# Case count: driven by "the previous day's case count" (continuity of transmission) + "temperature drive 7 days ago" + weekly rhythm + random noise
cases = np.zeros(n)
for i in range(n):
    lag = cases[i - 1] if i >= 1 else 0
    cases[i] = max(
        0,
        0.55 * lag                                   # continuity from the previous day's case count
        + 3.2 * (drive[i - 7] if i >= 7 else 0)       # delayed effect of temperature 7 days ago (the leading indicator)
        + 4 * np.sin(2 * np.pi * t[i] / 7)            # weekly reporting rhythm
        + 6                                            # baseline case count
        + np.random.normal(0, 2.0),                   # random noise
    )

print(f"Series length: {n} days")
print(f"Cases: mean={cases.mean():.1f}, min={cases.min():.1f}, max={cases.max():.1f}")
print(f"Temperature: mean={temp.mean():.1f}°C, min={temp.min():.1f}°C, max={temp.max():.1f}°C")

# -- Plot both time series so we can eyeball the relationship between temperature and case count --
fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.plot(t, cases, color="#D97757", label="Daily cases")
ax1.set_xlabel("Day index")
ax1.set_ylabel("Daily cases", color="#D97757")
ax1.tick_params(axis="y", labelcolor="#D97757")

ax2 = ax1.twinx()
ax2.plot(t, temp, color="#6A9BCC", alpha=0.7, label="Temperature (°C)")
ax2.set_ylabel("Temperature (°C)", color="#6A9BCC")
ax2.tick_params(axis="y", labelcolor="#6A9BCC")
ax2.grid(False)

fig.suptitle("Synthetic dengue × temperature series (teaching data)")
fig.tight_layout()
plt.show()

print("\n→ Look closely: doesn't the temperature peak arrive a few days before the case-count peak? That's the leading indicator.")

## Cutting "one long timeline" into "a pile of training samples"

A neural network needs fixed-shape inputs — you can't just feed it one 360-day-long sequence. The trick is a **sliding window**:

- Each sample: the past **L=21 days** of (case count, temperature) as input X, and the case count **7 days later (H=7)** as label y
- Shift the window forward by one day and you get the next sample — so neighboring samples overlap heavily, which is the norm for time-series data

### Split by time — never split randomly!

Ch10's `train_test_split(shuffle=True)` was fine because those rows were independent of each other. **Sequence data must never be split randomly** — if future windows leak into the training set, the model effectively "sees the answer," and offline performance will be badly overestimated.

The correct approach is to **split by time**:

1. The first 300 days become the "training window," and the last 60 days become the **test set** (the model never sees it)
2. Within the training window, carve out the **last 40 days as the validation set**, used for early stopping — again split by time, no shuffling
3. **The mean and standard deviation used for standardization must come only from the training window (the first 300 days)** — never peek at the test set. This is the basic principle for avoiding data leakage (echoing the train/val discipline from Ch10)

In [ ]:
# --- Step 2: sliding windows + time-based train/val/test split + standardize using training statistics only ---
H = 7   # how many days ahead to predict the case count
L = 21  # how many days of history each sample looks back over

# Stack the two features together: column 0 = cases, column 1 = temp
feats = np.stack([cases, temp], axis=1).astype(np.float32)

# split: the first 300 days are the "training window" (train+val are both carved from here); the last 60 days are a test set the model never sees
split = n - 60

# Standardization statistics may only be computed from the training window -- never peek at the test set (or you get data leakage)
mu = feats[:split].mean(axis=0)
sd = feats[:split].std(axis=0)
z = (feats - mu) / sd


def windows(s, e):
    """Cut the standardized series into sliding-window samples.

    For each i in [s, e-H):
      X = z[i-L : i]      → the past L days of (cases, temp)
      y = z[i+H-1, 0]     → the case count H days later (only the cases column)

    Returns (X array, y array, the corresponding original day indices); the indices can later be used to invert the standardization and line up against the true case counts.
    """
    Xs, ys, idxs = [], [], []
    for i in range(s, e - H):
        Xs.append(z[i - L:i])
        ys.append(z[i + H - 1, 0])
        idxs.append(i + H - 1)
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32), np.array(idxs)


# All windows produced from the training window (first 300 days)
Xtr_full, ytr_full, idxtr_full = windows(L, split)

# Time-based split: within the training window, the last 40 days become the validation set (no shuffle, split by time)
val_start = split - 40
train_mask = idxtr_full < val_start
Xtr, ytr = Xtr_full[train_mask], ytr_full[train_mask]
Xval, yval = Xtr_full[~train_mask], ytr_full[~train_mask]

# Test set: the last 60 days (the model has never seen them, and the standardization statistics never used them either)
Xte, yte, idxte = windows(split - H + 1, n + 1)

# Convert to tensors
Xtr_t, ytr_t = torch.tensor(Xtr), torch.tensor(ytr).unsqueeze(1)
Xval_t, yval_t = torch.tensor(Xval), torch.tensor(yval).unsqueeze(1)
Xtrfull_t, ytrfull_t = torch.tensor(Xtr_full), torch.tensor(ytr_full).unsqueeze(1)
Xte_t, yte_t = torch.tensor(Xte), torch.tensor(yte).unsqueeze(1)

print(f"train: {Xtr.shape[0]:>3d} windows (day {idxtr_full[train_mask].min()}-{idxtr_full[train_mask].max()})")
print(f"val  : {Xval.shape[0]:>3d} windows (day {idxtr_full[~train_mask].min()}-{idxtr_full[~train_mask].max()})")
print(f"test : {Xte.shape[0]:>3d} windows (day {idxte.min()}-{idxte.max()})")
print(f"Shape of each window: {Xtr.shape[1:]} = (L={L} days, 2 features)")

## LSTM = a detective with memory

An LSTM (Long Short-Term Memory) has a set of internal "memory cells" that decide, as they read through the sequence, "should this piece of information be kept, or forgotten?"

Think of it like a detective working a case: after reading 21 days' worth of clues, what's left in the LSTM's head isn't just a fragmentary memory of the last day — it's a **summary of the whole case**: "Has the case count been climbing all week? Has temperature stayed elevated for several days in a row?" — and it makes its 7-day-ahead prediction based on that summary.

### How does early stopping fit in?

Inside the training loop, we simultaneously use the **validation set** to monitor val MAE:

1. First let the model warm up for a while (the first 150 epochs) — during this period val performance naturally oscillates, and checking too early can be misled by noise
2. Only after warmup does the "patience countdown" begin: if val MAE fails to improve for 30 consecutive epochs, stop training and roll back to "the epoch with the best performance"
3. Once early stopping has picked "how many epochs to train," we retrain a final model for that same number of epochs on **all** of the pre-test data (train+val combined) — because in sequence forecasting, the few dozen days closest to the test period carry the most useful information, and they shouldn't be permanently excluded from the production model

In [ ]:
# --- Step 3: LSTM model + training loop (with early stopping) ---

class LSTMModel(nn.Module):
    """Reads the past L days of (cases, temp) with an LSTM and outputs a prediction for the case count H days later."""

    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=2, hidden_size=32, batch_first=True)  # 2 input features: cases, temp
        self.fc = nn.Linear(32, 1)  # turn the final hidden state into a single predicted value

    def forward(self, x):
        out, _ = self.lstm(x)      # out shape: (batch, L, 32) -- one hidden state per time step
        last_step = out[:, -1, :]  # take only the hidden state at the last time step, after reading the whole sequence
        return self.fc(last_step)


def train_with_early_stopping(model_cls, seed=1, max_epochs=200, patience=30, warmup=150):
    """Training loop + early stopping; returns the epoch number with the best performance.

    Approach:
      1. Each epoch updates the weights on the "training set" (Adam + MSELoss)
      2. Each epoch computes val MAE on the "validation set" to monitor whether the model is really learning something (rather than memorizing answers)
      3. During warmup (the first 150 epochs), let the model train steadily without making early-stopping decisions
      4. Only after warmup does the patience counter start: stop once val MAE hasn't improved for 30 consecutive epochs
      5. Return the best epoch number -- later we retrain the production model for that many epochs on all training data
    """
    torch.manual_seed(seed)
    model = model_cls()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.008)
    loss_fn = nn.MSELoss()

    best_val_mae = float("inf")
    counter = 0
    best_epoch = warmup - 1

    for epoch in range(max_epochs):
        # -- one training step --
        model.train()
        optimizer.zero_grad()
        pred = model(Xtr_t)
        loss = loss_fn(pred, ytr_t)
        loss.backward()
        optimizer.step()

        # -- monitor with the validation set --
        model.eval()
        with torch.no_grad():
            val_pred = model(Xval_t)
            val_mae = torch.mean(torch.abs(val_pred - yval_t)).item()

        if epoch < warmup:
            continue  # warmup period: train only, no early-stopping decision

        if val_mae < best_val_mae:
            best_val_mae, counter, best_epoch = val_mae, 0, epoch
        else:
            counter += 1
        if counter >= patience:
            print(f"  early stopping at epoch {epoch} (best epoch = {best_epoch})")
            break

    return best_epoch, best_val_mae


def refit_on_full(model_cls, n_epochs, seed=1):
    """Using the epoch count chosen by early stopping, retrain the final model on ALL of the pre-test data (train+val).

    Why the extra step? Because in sequence forecasting, the information in the few dozen days closest to the test period is the most valuable --
    if the production model only fit the training subset with the validation days removed, it would be throwing away the most recent information.
    Early stopping's only job is to decide "how many epochs to train"; once that's decided,
    we retrain for that same number of epochs on all available data -- a common practice in forecasting.
    """
    torch.manual_seed(seed)
    model = model_cls()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.008)
    loss_fn = nn.MSELoss()
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(Xtrfull_t)
        loss = loss_fn(pred, ytrfull_t)
        loss.backward()
        optimizer.step()
    return model


best_epoch_lstm, val_mae_lstm = train_with_early_stopping(LSTMModel)
print(f"LSTM: best epoch chosen by validation = {best_epoch_lstm} (val MAE, standardized scale = {val_mae_lstm:.4f})")

lstm_model = refit_on_full(LSTMModel, best_epoch_lstm + 1)
print(f"LSTM: retrained for {best_epoch_lstm + 1} epochs on all training data (train+val, {len(Xtr_full)} windows total)")

## 1D-CNN = spotting local fingerprints

Unlike the LSTM, which carries a running memory forward, a CNN (convolutional neural network) slides a small window (a kernel, width 3 here) across the whole sequence, specifically looking for **local shape features** — e.g. "case counts rose for 3 days in a row" or "temperature suddenly jumped over a few days" — like spotting a fingerprint: it doesn't matter which day of the window the shape appears on, as long as it shows up, it gets caught.

Stacking two convolutional layers lets "local shapes" combine into more complex patterns; the result is flattened and fed into a linear layer that outputs the prediction. Training works exactly the same way as the LSTM: we reuse the `train_with_early_stopping()` and `refit_on_full()` functions defined in Step 3, just swapping in the CNN model.

In [ ]:
# --- Step 4: 1D-CNN model + training (reusing the early-stopping functions from Step 3) ---

class CNNModel(nn.Module):
    """Two 1D convolutional layers capture local shapes; flatten and feed into a linear layer for the prediction."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels=2, out_channels=24, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv1d(in_channels=24, out_channels=24, kernel_size=3, padding=1), nn.ReLU(),
            nn.Flatten(),
        )
        self.fc = nn.Linear(24 * L, 1)

    def forward(self, x):
        # Conv1d expects input shaped (batch, channels, sequence length),
        # but our x is (batch, sequence length L, 2 features), so we transpose first
        x = x.transpose(1, 2)
        return self.fc(self.net(x))


best_epoch_cnn, val_mae_cnn = train_with_early_stopping(CNNModel)
print(f"CNN: best epoch chosen by validation = {best_epoch_cnn} (val MAE, standardized scale = {val_mae_cnn:.4f})")

cnn_model = refit_on_full(CNNModel, best_epoch_cnn + 1)
print(f"CNN: retrained for {best_epoch_cnn + 1} epochs on all training data (train+val, {len(Xtr_full)} windows total)")

## How do we know if the model is actually useful? -- first beat "doing nothing"

The absolute value of MAE by itself is meaningless; it has to be compared against an **honest floor**:

- **Persistence (naive) baseline**: simply assume "the case count 7 days from now = the most recently known case count," using no model and no temperature information at all
- If LSTM / CNN can't beat this baseline, the model's added complexity was wasted -- this is the single most important first hurdle in evaluating sequence forecasts

Three evaluation metrics:

| Metric | Meaning |
|------|------|
| MAE (Mean Absolute Error) | Average error in case counts -- the most intuitive |
| RMSE (Root Mean Squared Error) | More sensitive to large errors -- a single outlier gets amplified |
| MAPE (Mean Absolute Percentage Error) | Error as a percentage of the true value -- easy to compare across scenarios |

In [ ]:
# --- Step 5: persistence baseline + results comparison table + test-window prediction plot ---

def inverse_transform_cases(z_pred):
    """Convert a standardized prediction back to the true case-count scale."""
    return z_pred * sd[0] + mu[0]


def mae_score(pred, idxs):
    return np.mean(np.abs(pred - cases[idxs]))


def rmse_score(pred, idxs):
    return np.sqrt(np.mean((pred - cases[idxs]) ** 2))


def mape_score(pred, idxs):
    true = cases[idxs]
    return np.mean(np.abs(pred - true) / np.maximum(true, 1e-6)) * 100


# Model predictions (converted back to the case-count scale)
with torch.no_grad():
    lstm_pred = inverse_transform_cases(lstm_model(Xte_t).squeeze(1).numpy())
    cnn_pred = inverse_transform_cases(cnn_model(Xte_t).squeeze(1).numpy())

# Persistence (naive) baseline: case count 7 days later = the most recently known case count, no model at all
persistence_pred = np.array([cases[j - H] for j in idxte])

results = {
    "Persistence (naive baseline)": persistence_pred,
    "LSTM": lstm_pred,
    "1D-CNN": cnn_pred,
}

print(f"{'Model':<28}{'MAE':>8}{'RMSE':>8}{'MAPE(%)':>10}")
print("-" * 54)
for name, pred in results.items():
    print(f"{name:<28}{mae_score(pred, idxte):>8.3f}{rmse_score(pred, idxte):>8.3f}{mape_score(pred, idxte):>10.2f}")

lstm_beats = mae_score(lstm_pred, idxte) < mae_score(persistence_pred, idxte)
cnn_beats = mae_score(cnn_pred, idxte) < mae_score(persistence_pred, idxte)
print(f"\nDoes LSTM beat persistence? {lstm_beats}")
print(f"Does CNN  beat persistence? {cnn_beats}")

# -- Plot predicted vs. true case counts over the test window --
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(idxte, cases[idxte], label="True cases", color="#1A1A1A", linewidth=2)
ax.plot(idxte, persistence_pred, label="Persistence (naive)", color="#6B6B6B", linestyle="--")
ax.plot(idxte, lstm_pred, label="LSTM", color="#D97757")
ax.plot(idxte, cnn_pred, label="1D-CNN", color="#6A9BCC")
ax.set_xlabel("Day index")
ax.set_ylabel("Daily new cases")
ax.set_title("Test window: 7-day-ahead case predictions -- true values vs. each model")
ax.legend()
plt.tight_layout()
plt.show()

## The honest conclusion

The LSTM / CNN in this section beat the persistence baseline, **not because deep learning is inherently stronger**, but because:

1. **They used temperature as a leading indicator** -- persistence can only see the case count itself; it has no way of knowing that temperature already rose 7 days ago and the case count is therefore likely to rise too
2. **The DGP contains a nonlinear, time-delayed interaction** (temperature 7 days ago driving today's case count), which is exactly the kind of pattern neural networks are good at capturing

### When is the naive baseline hard to beat?

If a curve is a **simple, smooth univariate series** (no extra leading indicator, no obvious nonlinear delay), persistence or a simple moving average is often already very strong, and complex models may not meaningfully beat it -- in that case, bolting on an LSTM/CNN may just add complexity and overfitting risk for nothing. In practice, whenever you face a sequence-forecasting problem, **always run a naive baseline first** before deciding whether to bring in a model.

### DL still needs "enough history"

The window length here (L=21 days) and the number of training samples (272 time windows) are both far more forgiving than the 280 cross-sectional rows used in Ch10/the first half of Ch11 -- sequence models inherently need a longer accumulated history to learn stable patterns. If a real-world outbreak surveillance series only has a few dozen days, deep learning is unlikely to have an advantage.

In the next chapter (Ch12), we return to the classification problem itself, but ask a different question: did showering "cause" the infections, or is it merely "associated" with them? → Causal inference.